# Energy Theft Detection — Analysis Notebook

This notebook walks through the full analysis: exploring the synthetic dataset,
applying the original rule-based detection method, and testing an Isolation
Forest model as an extension. See the project `README.md` for the real-world
context and data disclosure.


In [1]:
import sys
sys.path.insert(0, '../src/models')
sys.path.insert(0, '../src/features')

import pandas as pd
import matplotlib.pyplot as plt
from build_features import load_raw_data, build_feature_table
from rule_based import load_features_and_labels, flag_suspects, evaluate as evaluate_rule
from anomaly_detection import load_data, run_isolation_forest, evaluate as evaluate_ml

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 110


## 1. Load and inspect the raw data

In [2]:
consumption, alarms, meters = load_raw_data()
print(f"Meters: {len(meters)} | Consumption rows: {len(consumption)} | Alarm events: {len(alarms)}")
meters.head()


Meters: 500 | Consumption rows: 90000 | Alarm events: 327


,meter_id,zone_id,customer_type,contracted_capacity_kva
0,MTR-00001,ZONE-01,residential,15
1,MTR-00002,ZONE-08,commercial,25
2,MTR-00003,ZONE-07,industrial,150
3,MTR-00004,ZONE-05,industrial,150
4,MTR-00005,ZONE-05,residential,5


In [3]:
consumption.head()


,meter_id,date,kwh_consumed
0,MTR-00001,2025-01-01,11.23
1,MTR-00001,2025-01-02,11.55
2,MTR-00001,2025-01-03,12.29
3,MTR-00001,2025-01-04,10.40
4,MTR-00001,2025-01-05,11.09


In [4]:
alarms['alarm_type'].value_counts()


alarm_type
tilt               102
cover_open          91
magnetic_tamper     74
reverse_energy      60
Name: count, dtype: int64

## 2. Feature engineering

For each meter, compare a 60-day baseline period against the rest of the
series: average consumption, the steepest sustained 14-day drop, and counts
of each tamper alarm type.


In [5]:
features = build_feature_table(consumption, alarms, meters)
features.describe()


,contracted_capacity_kva,baseline_avg_kwh,monitoring_avg_kwh,monitoring_std_kwh,min_14d_rolling_kwh,alarm_count_cover_open,alarm_count_magnetic_tamper,alarm_count_reverse_energy,alarm_count_tilt,total_alarms,drop_ratio,min_drop_ratio
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,46.350000,82.300238,80.276042,10.055804,70.073927,0.182000,0.148000,0.120000,0.204000,0.654000,0.020178,0.143463
std,56.059905,106.849169,104.487052,14.182410,91.766759,0.491283,0.441005,0.349061,0.496868,1.103049,0.102797,0.134270
min,5.000000,7.864500,5.745583,0.641391,4.695000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.045737,-0.000732
25%,10.000000,15.363125,14.473208,1.736285,12.675000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.020340,0.060689
50%,15.000000,21.882833,21.888125,2.534186,19.173750,0.000000,0.000000,0.000000,0.000000,0.000000,-0.008713,0.102194
75%,75.000000,101.292750,102.454437,11.888273,88.436786,0.000000,0.000000,0.000000,0.000000,1.000000,0.000884,0.163874
max,200.000000,395.269167,400.344000,100.453418,367.575714,3.000000,3.000000,2.000000,3.000000,8.000000,0.581278,0.659471


## 3. Rule-based detection (original methodology)

Flag a meter if it shows BOTH a sustained consumption drop of 30%+ AND at
least one tamper alarm (tilt / magnetic tamper / cover open) in the
monitoring period.


In [6]:
df = load_features_and_labels()
df = flag_suspects(df)
rule_metrics = evaluate_rule(df)
rule_metrics


{'true_positives': 44,
 'false_positives': 0,
 'false_negatives': 6,
 'true_negatives': 450,
 'precision': 1.0,
 'recall': 0.88,
 'f1_score': 0.936,
 'flagged_count': 44}

## 4. Isolation Forest (ML extension)

The rule-based method is precise but misses cases where the alarm didn't
fire or wasn't logged. Isolation Forest looks at the full feature space and
can catch statistically unusual meters even without an explicit alarm.


In [7]:
ml_df = load_data()
ml_df = run_isolation_forest(ml_df)
ml_metrics = evaluate_ml(ml_df['is_fraud'], ml_df['ml_flagged'])
ml_metrics


{'true_positives': 40,
 'false_positives': 10,
 'false_negatives': 10,
 'true_negatives': 440,
 'precision': 0.8,
 'recall': 0.8,
 'f1_score': 0.8,
 'flagged_count': 50}

## 5. Comparing approaches

In [8]:
comparison = pd.DataFrame({
    'rule_based': rule_metrics,
    'isolation_forest': ml_metrics,
}).T[['precision', 'recall', 'f1_score', 'flagged_count', 'true_positives', 'false_positives', 'false_negatives']]
comparison


,precision,recall,f1_score,flagged_count,true_positives,false_positives,false_negatives
rule_based,1.0,0.88,0.936,44.0,44.0,0.0,6.0
isolation_forest,0.8,0.80,0.800,50.0,40.0,10.0,10.0


## 6. Visual results

![Consumption example](../outputs/consumption_example.png)

![Drop vs alarms](../outputs/drop_vs_alarms.png)

![Model comparison](../outputs/model_comparison.png)


## 7. Takeaways

- The rule-based method (drop + alarm) is highly precise — almost no false
  alarms sent to the field team — but conservative, missing theft cases
  where the tamper alarm wasn't triggered or logged.
- Isolation Forest catches additional cases by looking at consumption
  behavior alone, at the cost of more false positives to review manually.
- Combining both (flag if EITHER approach fires) maximizes recall for a
  human review queue — a common trade-off in fraud detection: better to
  send a field crew to check a few extra false leads than to miss a real
  theft case that keeps costing the utility money every month.
- See `docs/methodology.md` for the real-world context this project is
  based on, and `README.md` for the data disclosure.
